# Train & Evaluate the baseline for next-token prediciton

In [40]:
# imports

from pathlib import Path
import pickle
import torch
from torch.utils.data import Dataset, DataLoader
from data_utils import NextItemDataset, collate_batch
from transformers import GPT2Config, GPT2LMHeadModel
from torch.optim import AdamW
import time

## Load data

In [41]:
with open(
    Path("./../data") / "artifacts.pkl", "rb"
) as f:
    artifacts = pickle.load(f)

user_sequences = artifacts["user_sequences"]
item_to_token = artifacts["item_to_token"]
token_to_item = artifacts["token_to_item"]
train_sequences = artifacts["train_sequences"]

t_train = artifacts["t_train"]
t_validate = artifacts["t_validate"]

print("Users:", len(user_sequences))
print("Vocabulary size:", len(item_to_token))
print("Training sequences:", len(train_sequences))

Users: 50
Vocabulary size: 1000
Training sequences: 7920


In [42]:
# initialize the dataset & the dataloader for the current notebook
MAX_LENGTH = 32
BATCH_SIZE = 8

train_dataset = NextItemDataset(
    train_sequences,
    max_length=MAX_LENGTH,
)

def collate_fn(seqs):
    return collate_batch(seqs, len(item_to_token))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
print(f"We have {len(train_dataset)} chunks for training")

We have 8191 chunks for training


## Model Initialization

In [43]:
N_ITEMS = len(item_to_token)
PAD_ID = N_ITEMS

print(N_ITEMS)

MAX_LENGTH = 32

config = GPT2Config(
    vocab_size=N_ITEMS + 1,
    n_positions=MAX_LENGTH,
    n_ctx=MAX_LENGTH,
    n_embd=64, # dimensions for the embeddings
    n_layer=2,
    n_head=2,
    pad_token_id=PAD_ID,
    bos_token_id=None,
    eos_token_id=None,
)

model = GPT2LMHeadModel(config)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device) # move parameters to a chosen computing device

print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

1000
Device: mps
Parameters: 166208


In [44]:
# a small test
batch = next(iter(train_loader))

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

model.train() # enable training mode

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"],
)

print("Logits shape:", outputs.logits.shape)
print("Initial loss:", outputs.loss.item())

Logits shape: torch.Size([8, 22, 1001])
Initial loss: 6.900596618652344


## Baseline Model Traning

In [45]:
optimizer = AdamW(model.parameters(), lr=3e-4)

MAX_STEPS = 3000
LOG_EVERY = 25

model.train()
step = 0
start_time = time.time()

for epoch in range(3):
    for batch in train_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()
        step += 1

        if step % LOG_EVERY == 0:
            elapsed = time.time() - start_time
            print(
                f"Step {step} | "
                f"Loss {loss.item():.4f} | "
                f"Elapsed {elapsed:.1f}s"
            )

        if step >= MAX_STEPS:
            break

    if step >= MAX_STEPS:
        break

print(f"Training finished in {time.time() - start_time:.1f}s")

Step 25 | Loss 6.8330 | Elapsed 1.7s
Step 50 | Loss 6.8807 | Elapsed 2.5s
Step 75 | Loss 6.9730 | Elapsed 3.2s
Step 100 | Loss 6.7110 | Elapsed 3.9s
Step 125 | Loss 6.5988 | Elapsed 4.7s
Step 150 | Loss 6.8101 | Elapsed 5.5s
Step 175 | Loss 6.6873 | Elapsed 6.1s
Step 200 | Loss 6.5461 | Elapsed 6.7s
Step 225 | Loss 6.6647 | Elapsed 7.4s
Step 250 | Loss 6.7167 | Elapsed 8.0s
Step 275 | Loss 6.6736 | Elapsed 8.7s
Step 300 | Loss 6.3066 | Elapsed 9.3s
Step 325 | Loss 5.9875 | Elapsed 9.8s
Step 350 | Loss 6.3917 | Elapsed 10.5s
Step 375 | Loss 6.2262 | Elapsed 11.0s
Step 400 | Loss 6.1755 | Elapsed 11.7s
Step 425 | Loss 6.3116 | Elapsed 12.2s
Step 450 | Loss 6.7662 | Elapsed 12.8s
Step 475 | Loss 6.6121 | Elapsed 13.5s
Step 500 | Loss 5.9231 | Elapsed 14.1s
Step 525 | Loss 6.0217 | Elapsed 14.8s
Step 550 | Loss 5.7518 | Elapsed 15.3s
Step 575 | Loss 5.9119 | Elapsed 15.9s
Step 600 | Loss 6.2053 | Elapsed 16.5s
Step 625 | Loss 6.2686 | Elapsed 17.1s
Step 650 | Loss 5.6632 | Elapsed 17.7s
St

## Save the pre-trained model

In [46]:
path = Path("./../data/bs_gpt2_rec")
model.save_pretrained(path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]